In [10]:

from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais
from pipelines.readers.cvm_balanco_patrimonial import ReaderCSVCVMBalancoPatrimonial

from yfinance import download
from numpy import nan, arange, polyfit, polyval
from pandas import DataFrame, set_option, concat, DateOffset
from datetime import date
from dateutil.relativedelta import relativedelta

In [11]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-07,VIVT3,TELEF BRASIL,ON EJ,707125712,1.169
69,IBEP,2026-07-07,WEGE3,WEG,ON NM,1485954732,3.293
70,IBEP,2026-07-07,YDUQ3,YDUQS PART,ON NM,261365845,0.108


In [12]:
tickers = list(df_b3_segmentos_setoriais["Código"]) + ["^BVSP"]

In [13]:

reader_csv_balanco_patrimonial = ReaderCSVCVMBalancoPatrimonial()

data = {}

benchmark = download("^BVSP", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)[["Adj Close"]].pct_change()

for codigo in tickers:
    
    # Balanco Patrimonial
    try:
        ativo_total = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="BPA_ind", cd_conta="1").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        ativo_total = nan
    try:
        resultado_bruto = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="DRE_ind", cd_conta="3.03").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        resultado_bruto = nan    
    try:
        receita_de_vendas = reader_csv_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="DRE_ind", cd_conta="3.01").astype(int).pct_change().iloc[:, 1].iloc[-1]
    except Exception:
        receita_de_vendas = nan

    try:
        serie_precos = download(codigo + ".SA" if codigo != "^BVSP" else codigo, period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)
    except Exception:
        
        drawdown_atual = nan
        residuo_reg_lin = nan

        continue
    
    # Drawdown
    preco_maximo = serie_precos['Adj Close'].max()
    preco_atual = serie_precos['Adj Close'].iloc[-1]
    drawdown_atual = (preco_atual - preco_maximo) / preco_maximo * 100
    
    # Regressão linear
    y = serie_precos["Adj Close"].to_numpy()
    x = arange(len(y))

    coef = polyfit(x, y, 1)      
    tendencia = polyval(coef, x)

    residuo_reg_lin = ((serie_precos["Adj Close"] - tendencia) / tendencia) * 100
    residuo_reg_lin = residuo_reg_lin.iloc[-1]
    
    # Beta
    beta = serie_precos["Adj Close"].pct_change().cov(benchmark["Adj Close"]) / benchmark["Adj Close"].var()
    
    # Retorno agrupado por mês
    serie_precos['ret'] = serie_precos["Adj Close"].pct_change()
    serie_precos['mes'] = serie_precos.index.month
    serie_precos['ano'] = serie_precos.index.year

    tabela = (
        serie_precos.groupby(['ano', 'mes'])[['ret']]
        .sum()
        .reset_index()
        .pivot(index='ano', columns='mes', values='ret')
    ) * 100
    
    data[codigo] = {
        "Ativo Total": ativo_total,
        "Resultado Bruto": resultado_bruto,
        "Receita de Vendas": receita_de_vendas,
        "Drawdown Atual": drawdown_atual,
        "Resíduo Regressão Linear": residuo_reg_lin,
        "Beta": beta,
        f"Retorno Agrupado por Mês (Média Mês:{date.today().month})": tabela[date.today().month].mean(),
        f"Retorno Agrupado por Mês (Média Mês:{(date.today() + relativedelta(months=1)).month})": tabela[(date.today() + relativedelta(months=1)).month].mean()
        }
    
    

In [16]:
df_final = (
    df_b3_segmentos_setoriais.set_index("Código")
    .join(DataFrame.from_dict(data, orient="index"))
    .reset_index()
)

benchmark_row = DataFrame([{
    "Código": "^BVSP",
    "Ativo Total": nan,
    "Resultado Bruto": nan,
    "Receita de Vendas": nan,
    "Drawdown Atual": data["^BVSP"]["Drawdown Atual"],
    "Resíduo Regressão Linear": data["^BVSP"]["Resíduo Regressão Linear"],
    "Beta": data["^BVSP"]["Beta"],
    f"Retorno Agrupado por Mês (Média Mês:{date.today().month})": data["^BVSP"][f"Retorno Agrupado por Mês (Média Mês:{date.today().month})"],
    f"Retorno Agrupado por Mês (Média Mês:{(date.today() + relativedelta(months=1)).month})": data["^BVSP"][f"Retorno Agrupado por Mês (Média Mês:{(date.today() + relativedelta(months=1)).month})"]
}])

df_final = concat([df_final, benchmark_row], ignore_index=True)

set_option("display.max_rows", None)
df_final =  df_final.sort_values(by="Drawdown Atual", ascending=False)
df_final

,Código,Indice,Data Referencia,Ação,Tipo,Qtde. Teórica,Part. (%),Ativo Total,Resultado Bruto,Receita de Vendas,Drawdown Atual,Resíduo Regressão Linear,Beta,Retorno Agrupado por Mês (Média Mês:7),Retorno Agrupado por Mês (Média Mês:8)
48,PSSA3,IBEP,2026-07-07,PORTO SEGURO,ON NM,1.788254e+08,0.457,0.056534,NaN,NaN,-4.785843,33.482209,0.539791,3.395100,5.421270
26,ENEV3,IBEP,2026-07-07,ENEVA,ON NM,1.913352e+09,2.392,0.041386,-0.340257,-0.268461,-7.357741,34.314900,0.747984,4.569565,4.708633
0,ABEV3,IBEP,2026-07-07,AMBEV S/A,ON,4.273841e+09,3.251,-0.025933,-0.071207,-0.043674,-7.520942,31.639336,0.650159,1.928327,-0.759528
59,TAEE11,IBEP,2026-07-07,TAESA,UNT N2,2.185682e+08,0.421,0.035111,0.030653,0.020434,-7.645196,7.885899,0.449320,4.269298,1.427716
62,UGPA3,IBEP,2026-07-07,ULTRAPAR,ON NM,1.067870e+09,1.429,0.045903,NaN,NaN,-8.895410,40.134033,1.195599,0.179374,0.262500
36,ITSA4,IBEP,2026-07-07,ITAUSA,PN N1,5.970738e+09,3.850,0.028099,NaN,NaN,-9.000617,39.835087,0.968049,3.541159,2.173550
31,GOAU4,IBEP,2026-07-07,GERDAU MET,PN N1,8.308378e+08,0.383,-0.011312,0.288102,-0.209715,-9.176910,8.016687,1.190893,8.805200,3.311804
18,CPLE3,IBEP,2026-07-07,COPEL,ON NM,2.969407e+09,2.123,0.066665,-0.061209,-0.055854,-10.523730,36.145997,0.837719,3.430206,2.515005
66,VBBR3,IBEP,2026-07-07,VIBRA,ON NM,1.192005e+09,1.720,0.043912,0.248137,-0.049013,-11.222554,40.858821,1.171303,3.609765,5.750550
30,GGBR4,IBEP,2026-07-07,GERDAU,PN N1,1.258208e+09,1.316,-0.011312,0.288102,-0.209715,-11.399592,2.257778,1.117174,7.993405,1.395292


In [2]:
df_final[df_final["Part. (%)"] > 5]

NameError: name 'df_final' is not defined